# ACEC CA Members Import
This notebook imports members from `data/ACEC_CA_Members.csv` into the `company` and `client` tables.

In [ ]:
!pip install pandas psycopg2-binary python-dotenv

In [3]:
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Database connection parameters
DB_HOST = os.getenv('DB_HOST', '192.168.86.100')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'aquiferpe')
DB_USER = os.getenv('DB_USER', 'aquifer_app')
DB_PASSWORD = os.getenv('DB_PASSWORD')

if not DB_PASSWORD:
    print("Warning: DB_PASSWORD not set in environment. Please set it in .env or here.")

def get_db_connection():
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )
    return conn

print("Database configuration loaded.")

Database configuration loaded.


In [4]:
# Load CSV Data
csv_file = 'ACEC_CA_Members.csv'
df = pd.read_csv(csv_file)

# Display first few rows
df.head()

,name,chapter,email,phone,website,address,desc
0,"Materials Testing, Inc. dba KC Engineering Com...",Non-Chapter,dcymanski@kcengr.com,707 447-4025 (Phone); 707 447-4143 (Fax),http://www.mti-kcgeotech.com,"865 Cotting Lane, Suite A, Vacaville, Californ...","Geotechnical engineering, special inspection, ..."
1,"R.E.Y. Engineers, Inc.",Sierra Chapter,NaN,NaN,NaN,"905 Sutter Street, Suite 200, Folsom, California","Based in Folsom, CA, R.E.Y. Engineers focuses ..."
2,Hillwig-Goodrow,Riverside/San Bernardino,hg@hillwig-goodrow.com,909 794-2673 (Phone); 909 794-2863 (Fax),http://www.hillwig-goodrow.com,"31419 Outer Highway 10, Suite I-200, Redlands,...",Hillwig-Goodrow provides Land Surveying servic...
3,"Galloway & Company, Inc.",San Joaquin Valley,terramortensen@gallowayus.com,303 7708884 (Phone),NaN,"575 East locust Avenue, Suite 103,, Fresno, Ca...",Galloway provides superior development solutio...
4,WSP USA,Non-Chapter,bruce.corkle@wsp.com,909 273-7400 (Phone); 909 273-7420 (Fax),http://www.wsp.com,"1250 N. Lakeview Avenue, Anaheim, California, ...","Consulting hydrogeologists, environmental and ..."


## Pass 1: Import Companies

In [ ]:
try:
    conn = get_db_connection()
    cur = conn.cursor()

    companies_added = 0
    errors = 0

    for index, row in df.iterrows():
        try:
            # Process Company
            company_name = row['name']
            if pd.isna(company_name):
                continue
                
            # Clean data
            location = row['address'] if not pd.isna(row['address']) else None
            overview = row['desc'] if not pd.isna(row['desc']) else None
            website = row['website'] if not pd.isna(row['website']) else None
            
            # Check if company exists
            cur.execute("SELECT id FROM company WHERE company_name = %s", (company_name,))
            res = cur.fetchone()
            
            if not res:
                # Create Company
                cur.execute("""
                    INSERT INTO company (company_name, location, overview, website)
                    VALUES (%s, %s, %s, %s)
                    """,
                    (company_name, location, overview, website)
                )
                companies_added += 1
                
        except Exception as e:
            print(f"Error processing row {index}: {e}")
            errors += 1
            conn.rollback()
            continue
            
    conn.commit()
    print(f"Finished! Companies added: {companies_added}, Errors: {errors}")
    
except Exception as e:
    print(f"Critical Error: {e}")
finally:
    if 'cur' in locals(): cur.close()
    if 'conn' in locals(): conn.close()


## Pass 2: Update Company Contact Info

In [ ]:
# Ensure columns exist
try:
    conn = get_db_connection()
    cur = conn.cursor()
    
    # Add email column if not exists
    cur.execute("SELECT column_name FROM information_schema.columns WHERE table_name='company' AND column_name='email'")
    if not cur.fetchone():
        print("Adding email column to company table...")
        cur.execute("ALTER TABLE company ADD COLUMN email VARCHAR(255)")
        
    # Add phone column if not exists
    cur.execute("SELECT column_name FROM information_schema.columns WHERE table_name='company' AND column_name='phone'")
    if not cur.fetchone():
        print("Adding phone column to company table...")
        cur.execute("ALTER TABLE company ADD COLUMN phone VARCHAR(255)")
        
    conn.commit()
    print("Schema check complete.")
except Exception as e:
    print(f"Schema Error: {e}")
    conn.rollback()
finally:
    if 'cur' in locals(): cur.close()
    if 'conn' in locals(): conn.close()

# Update Data
def clean_phone_number(phone_str):
    if pd.isna(phone_str):
        return None
    # Take the part before "(Phone"
    if "(Phone" in str(phone_str):
        return str(phone_str).split("(Phone")[0].strip()
    return str(phone_str).strip()

try:
    conn = get_db_connection()
    cur = conn.cursor()
    
    updated_count = 0
    errors = 0
    
    print("Updating company records...")
    
    for index, row in df.iterrows():
        try:
            company_name = row['name']
            if pd.isna(company_name):
                continue
            
            # Get email
            email = row['email'] if not pd.isna(row['email']) else None
            
            # Get and clean phone
            phone = clean_phone_number(row['phone']) if not pd.isna(row['phone']) else None
            
            if email or phone:
                # Update query
                cur.execute("""
                    UPDATE company 
                    SET email = COALESCE(%s, email), 
                        phone = COALESCE(%s, phone)
                    WHERE company_name = %s
                """, (email, phone, company_name))
                
                if cur.rowcount > 0:
                    updated_count += 1
                    
        except Exception as e:
            print(f"Error updating row {index}: {e}")
            errors += 1
            conn.rollback()
            continue
            
    conn.commit()
    print(f"Update Finished! Companies updated: {updated_count}, Errors: {errors}")
    
except Exception as e:
    print(f"Critical Error: {e}")
finally:
    if 'cur' in locals(): cur.close()
    if 'conn' in locals(): conn.close()